# Process Reward Modeling

Author xiaodongguaAIGC

In [59]:
import torch
import torch.nn as nn

## step datasets

In [60]:
vocab_size = 100
step_length = 100

In [62]:
# CoT data
# A is prompt, B D E is correct result
# for
A = torch.randint(high = vocab_size, size=(1, step_length))
B = torch.randint(high = vocab_size, size=(1, step_length))
C = torch.randint(high = vocab_size, size=(1, step_length))
D = torch.randint(high = vocab_size, size=(1, step_length))
E = torch.randint(high = vocab_size, size=(1, step_length))

data =  [[B, C], [C, D, E], [B, D, E]]
label = [[1, 0], [0, 1, 1], [1, 1, 1]]

data_list = []
for cot in data:
    current = [A]
    for step in cot:
        current.append(A)
        prompt = torch.cat(current, dim=1)
        data_list.append(prompt)
# print(len(data_list))
print(data_list)
label_list = [ data for list in label for data in list]
print(label_list)

[
    tensor([[8, 5, 2, 4, 5, 6, 9, 3, 1, 0, 8, 5, 2, 4, 5, 6, 9, 3, 1, 0]]),
    tensor([[8, 5, 2, 4, 5, 6, 9, 3, 1, 0, 8, 5, 2, 4, 5, 6, 9, 3, 1, 0, 8, 5, 2, 4,
         5, 6, 9, 3, 1, 0]]),
    tensor([[8, 5, 2, 4, 5, 6, 9, 3, 1, 0, 8, 5, 2, 4, 5, 6, 9, 3, 1, 0]]),
    tensor([[8, 5, 2, 4, 5, 6, 9, 3, 1, 0, 8, 5, 2, 4, 5, 6, 9, 3, 1, 0, 8, 5, 2, 4,
         5, 6, 9, 3, 1, 0]]),
    tensor([[8, 5, 2, 4, 5, 6, 9, 3, 1, 0, 8, 5, 2, 4, 5, 6, 9, 3, 1, 0, 8, 5, 2, 4,
         5, 6, 9, 3, 1, 0, 8, 5, 2, 4, 5, 6, 9, 3, 1, 0]]),
    tensor([[8, 5, 2, 4, 5, 6, 9, 3, 1, 0, 8, 5, 2, 4, 5, 6, 9, 3, 1, 0]]),
    tensor([[8, 5, 2, 4, 5, 6, 9, 3, 1, 0, 8, 5, 2, 4, 5, 6, 9, 3, 1, 0, 8, 5, 2, 4,
         5, 6, 9, 3, 1, 0]]),
    tensor([[8, 5, 2, 4, 5, 6, 9, 3, 1, 0, 8, 5, 2, 4, 5, 6, 9, 3, 1, 0, 8, 5, 2, 4,
         5, 6, 9, 3, 1, 0, 8, 5, 2, 4, 5, 6, 9, 3, 1, 0]])
]

[1, 0, 0, 1, 1, 1, 1, 1]

# Process Reward Model

In [63]:
class PRM(nn.Module):
    def __init__(self, vocab_size=100, embd_size=128, num_class=2):
        super(PRM, self).__init__()
        self.embd = nn.Embedding(vocab_size, embd_size)
        self.wq = nn.Linear(embd_size, embd_size)
        self.wk = nn.Linear(embd_size, embd_size)
        self.wv = nn.Linear(embd_size, embd_size)
        self.wo = nn.Linear(embd_size, embd_size)
        self.head = nn.Linear(embd_size, 2)
        self.log_softmax = nn.LogSoftmax(dim = 1)
    def forward(self, x):
        x = self.embd(x)
        q,k,v = self.wq(x), self.wk(x), self.wv(x)
        attn = q@k.t()@v
        last_hidden_states = self.wo(attn)
        logits = self.head(last_hidden_states)
        logprob = self.log_softmax(logits)
        return {'last_hidden_states' : last_hidden_states, 
                'logits' : logits,
                'logprob' : logprob}

model = PRM(vocab_size)

## PRM Train

In [64]:
import torch.optim as optim
optimizer = optim.SGD(model.parameters(), lr=0.00001)  # Adam优化器
# loss_fn = nn.CrossEntropyLoss()  # 均方误差损失
for i in range(10): # epochs
    optimizer.zero_grad()
    loss_acc=0
    for data, label in zip(data_list, label_list):
        logprob = model(data[0])['logprob']
        loss = -logprob[-1,label]
        loss_acc+=loss.item()
        # optimizer.zero_grad()
        loss.backward()
    print(loss)
    optimizer.step()

tensor(0.0010, grad_fn=<NegBackward0>)

tensor(0.2125, grad_fn=<NegBackward0>)

tensor(0.0719, grad_fn=<NegBackward0>)

tensor(0.4792, grad_fn=<NegBackward0>)

tensor(0.0032, grad_fn=<NegBackward0>)

tensor(0.4162, grad_fn=<NegBackward0>)

tensor(0.0065, grad_fn=<NegBackward0>)

tensor(0.5565, grad_fn=<NegBackward0>)

tensor(0.0017, grad_fn=<NegBackward0>)

tensor(0.2597, grad_fn=<NegBackward0>)